# **Inspección Automatizada de Estructuras mediante Visión Computacional**
**Resumen:** Detección de fisuras en concreto con OpenCV y comparación con Deep Learning (Hugging Face). Incluye preprocesamiento, umbralización, Canny, clasificación condicional y análisis técnico normativo (ACI 224R).


## **0. Instalación e Importación de Librerías**
Instalación de dependencias para visión computacional y modelos preentrenados.


In [ ]:
!pip install -q transformers torch torchvision pillow requests matplotlib

import os
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests

print(f'OpenCV: {cv2.__version__} | NumPy: {np.__version__}')


## **Parte 1 - Paso 1: Carga de Fotografías y Google Drive**
Carga de dos imágenes de concreto: una con fisura evidente y otra con ruido visual (manchas/sombras). Descarga automática si no están en Drive.


In [ ]:
# Montaje opcional de Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

RUTA_DRIVE_FISURA = '/content/drive/MyDrive/concreto_fisura.jpg'
RUTA_DRIVE_RUIDO  = '/content/drive/MyDrive/concreto_ruido.jpg'
ruta_fisura = 'concreto_fisura.jpg'
ruta_ruido  = 'concreto_ruido.jpg'

# Cargar desde Drive o descargar muestras reales de inspección
if os.path.exists(RUTA_DRIVE_FISURA) and os.path.exists(RUTA_DRIVE_RUIDO):
    shutil.copy(RUTA_DRIVE_FISURA, ruta_fisura)
    shutil.copy(RUTA_DRIVE_RUIDO, ruta_ruido)
else:
    url_f = 'https://raw.githubusercontent.com/CCNYRoboticsLab/concreteIn_inpection_VGGF/master/resultImages/labeled_images/crack/045.jpg'
    url_r = 'https://raw.githubusercontent.com/CCNYRoboticsLab/concreteIn_inpection_VGGF/master/resultImages/exampleImages/001.jpg'
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        with open(ruta_fisura, 'wb') as f: f.write(requests.get(url_f, headers=headers, timeout=10).content)
        with open(ruta_ruido, 'wb') as f: f.write(requests.get(url_r, headers=headers, timeout=10).content)
    except Exception:
        np.random.seed(42)
        h, w = 350, 450
        bg_f = np.random.normal(160, 15, (h, w)).clip(100, 220).astype(np.uint8)
        pts = np.array([[50, 40], [120, 100], [200, 180], [270, 270], [390, 340]], np.int32)
        cv2.polylines(bg_f, [pts], False, 35, 3)
        cv2.imwrite(ruta_fisura, cv2.cvtColor(bg_f, cv2.COLOR_GRAY2BGR))
        bg_r = np.random.normal(165, 12, (h, w)).clip(110, 220).astype(np.uint8)
        cv2.circle(bg_r, (220, 170), 80, 70, -1)
        bg_r = cv2.GaussianBlur(bg_r, (41, 41), 12)
        cv2.imwrite(ruta_ruido, cv2.cvtColor(bg_r, cv2.COLOR_GRAY2BGR))

img_fisura_bgr = cv2.imread(ruta_fisura)
img_ruido_bgr  = cv2.imread(ruta_ruido)
img_fisura_rgb = cv2.cvtColor(img_fisura_bgr, cv2.COLOR_BGR2RGB)
img_ruido_rgb  = cv2.cvtColor(img_ruido_bgr,  cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(img_fisura_rgb)
axes[0].set_title('1. Concreto con Fisura', fontweight='bold')
axes[0].axis('off')
axes[1].imshow(img_ruido_rgb)
axes[1].set_title('2. Concreto con Ruido (Manchas/Sombras)', fontweight='bold')
axes[1].axis('off')
plt.tight_layout()
plt.show()


## **Parte 1 - Paso 2: Escala de Grises y Suavizado Gaussiano**
Conversión a escala de grises para analizar intensidad y filtro Gaussiano (5x5) para reducir ruido de textura.


In [ ]:
# Escala de grises y suavizado Gaussiano (5x5)
gris_fisura = cv2.cvtColor(img_fisura_bgr, cv2.COLOR_BGR2GRAY)
gris_ruido  = cv2.cvtColor(img_ruido_bgr,  cv2.COLOR_BGR2GRAY)

blur_fisura = cv2.GaussianBlur(gris_fisura, (5, 5), 0)
blur_ruido  = cv2.GaussianBlur(gris_ruido,  (5, 5), 0)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes[0, 0].imshow(gris_fisura, cmap='gray'); axes[0, 0].set_title('Fisura - Gris'); axes[0, 0].axis('off')
axes[0, 1].imshow(blur_fisura, cmap='gray'); axes[0, 1].set_title('Fisura - Suavizado (5x5)'); axes[0, 1].axis('off')
axes[1, 0].imshow(gris_ruido, cmap='gray');  axes[1, 0].set_title('Ruido - Gris');  axes[1, 0].axis('off')
axes[1, 1].imshow(blur_ruido, cmap='gray');  axes[1, 1].set_title('Ruido - Suavizado (5x5)');  axes[1, 1].axis('off')
plt.tight_layout()
plt.show()


## **Parte 1 - Paso 3: Umbralización y Máscara (cv2.threshold)**
Binarización inversa con umbral 90: píxeles oscuros $\le 90$ (fisuras) pasan a 255 (blanco) y el fondo a 0 (negro).


In [ ]:
# Umbralización binaria inversa con umbral 90
UMBRAL = 90
_, mascara_fisura = cv2.threshold(blur_fisura, UMBRAL, 255, cv2.THRESH_BINARY_INV)
_, mascara_ruido  = cv2.threshold(blur_ruido,  UMBRAL, 255, cv2.THRESH_BINARY_INV)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(mascara_fisura, cmap='gray'); axes[0].set_title(f'Máscara Fisura (T={UMBRAL})'); axes[0].axis('off')
axes[1].imshow(mascara_ruido, cmap='gray');  axes[1].set_title(f'Máscara Ruido (T={UMBRAL})');  axes[1].axis('off')
plt.tight_layout()
plt.show()


## **Parte 1 - Paso 4: Detección de Bordes con cv2.Canny**
Detección de bordes y gradientes con umbrales de histéresis 50 y 150.


In [ ]:
# Detección de bordes Canny (50, 150)
bordes_fisura = cv2.Canny(blur_fisura, 50, 150)
bordes_ruido  = cv2.Canny(blur_ruido,  50, 150)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(bordes_fisura, cmap='inferno'); axes[0].set_title('Bordes Canny - Fisura'); axes[0].axis('off')
axes[1].imshow(bordes_ruido, cmap='inferno');  axes[1].set_title('Bordes Canny - Ruido');  axes[1].axis('off')
plt.tight_layout()
plt.show()


## **Parte 1 - Pasos 5 y 6: Porcentaje de Píxeles y Condicional `if`**
Cálculo del % de fisuración. Clasificación con umbral del 0.5%:

* `> 0.5%`: *"Posible fisura detectada. Revisar visualmente."*

* `≤ 0.5%`: *"Sin fisuras evidentes."*


In [ ]:
def evaluar_imagen(nombre, mascara, umbral_alerta=0.5):
    pixeles = cv2.countNonZero(mascara)
    total = mascara.shape[0] * mascara.shape[1]
    porcentaje = (pixeles / total) * 100.0
    
    if porcentaje > umbral_alerta:
        mensaje = 'Posible fisura detectada. Revisar visualmente.'
    else:
        mensaje = 'Sin fisuras evidentes.'
        
    print(f'{nombre}: {pixeles:,}/{total:,} px ({porcentaje:.3f}%) -> {mensaje}')
    return porcentaje, mensaje

pct_f, msg_f = evaluar_imagen('Estructura 1 (Fisura)', mascara_fisura)
pct_r, msg_r = evaluar_imagen('Estructura 2 (Ruido)', mascara_ruido)


In [ ]:
# Superposición de máscara sobre imagen original
def superponer(img_rgb, mascara):
    res = img_rgb.copy()
    idx = (mascara == 255)
    res[idx] = cv2.addWeighted(img_rgb[idx], 0.6, np.full_like(img_rgb[idx], [255, 0, 0]), 0.4, 0)
    return res

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(superponer(img_fisura_rgb, mascara_fisura))
axes[0].set_title(f'Fisura: {pct_f:.2f}% | {msg_f}', color='red' if pct_f > 0.5 else 'green', fontsize=10)
axes[0].axis('off')
axes[1].imshow(superponer(img_ruido_rgb, mascara_ruido))
axes[1].set_title(f'Ruido: {pct_r:.2f}% | {msg_r}', color='red' if pct_r > 0.5 else 'green', fontsize=10)
axes[1].axis('off')
plt.tight_layout()
plt.show()


## **Experimento: Sensibilidad del Umbral (40 vs 90 vs 150)**
Comparación del impacto de variar el umbral de detección sobre fisuras y ruido.


In [ ]:
# Prueba comparativa con umbrales 40, 90 y 150
umbrales = [40, 90, 150]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for col, t in enumerate(umbrales):
    _, m_f = cv2.threshold(blur_fisura, t, 255, cv2.THRESH_BINARY_INV)
    _, m_r = cv2.threshold(blur_ruido,  t, 255, cv2.THRESH_BINARY_INV)
    p_f = (cv2.countNonZero(m_f) / m_f.size) * 100
    p_r = (cv2.countNonZero(m_r) / m_r.size) * 100
    
    axes[0, col].imshow(m_f, cmap='gray'); axes[0, col].set_title(f'Fisura T={t}: {p_f:.2f}%'); axes[0, col].axis('off')
    axes[1, col].imshow(m_r, cmap='gray'); axes[1, col].set_title(f'Ruido T={t}: {p_r:.2f}%'); axes[1, col].axis('off')

plt.suptitle('Efecto del Umbral en la Detección', fontweight='bold')
plt.tight_layout()
plt.show()


## **Parte 1 - Paso 7: Modelo de Deep Learning en Hugging Face**
Inferencia con el modelo preentrenado `dima806/surface_crack_image_detection` (Vision Transformer ViT) para evaluar robustez frente al ruido.


In [ ]:
# Inferencia con Vision Transformer de Hugging Face
from transformers import pipeline

try:
    clasificador = pipeline('image-classification', model='dima806/surface_crack_image_detection')
    pred_f = clasificador(Image.open(ruta_fisura).convert('RGB'))
    pred_r = clasificador(Image.open(ruta_ruido).convert('RGB'))
    
    print(f'HF Fisura: {pred_f[0]["label"]} ({pred_f[0]["score"]*100:.2f}%)')
    print(f'HF Ruido:  {pred_r[0]["label"]} ({pred_r[0]["score"]*100:.2f}%)')
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img_fisura_rgb); axes[0].set_title(f'HF Fisura: {pred_f[0]["label"]} ({pred_f[0]["score"]*100:.1f}%)'); axes[0].axis('off')
    axes[1].imshow(img_ruido_rgb);  axes[1].set_title(f'HF Ruido: {pred_r[0]["label"]} ({pred_r[0]["score"]*100:.1f}%)'); axes[1].axis('off')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Aviso Hugging Face: {e}')


# **Parte 2: Preguntas Abiertas de Análisis (Resumen)**

### **1. Análisis del Umbral (40 vs 90 vs 150)**
* **$T = 150$:** Acepta tonos medios del concreto sano; la máscara se satura de ruido blanco ($20\% - 40\%$) generando **falsos positivos masivos**.
* **$T = 40$:** Solo capta negros profundos; microfisuras y ramificaciones delgadas desaparecen ($< 0.05\%$) generando **falsos negativos críticos**.
* **Valor óptimo ($T = 90$):** Se sitúa en el valle bimodal del histograma que separa la sombra de la fisura de la reflectancia del concreto sano.

### **2. Limitaciones en Obra con Drones**
* **Falsos positivos:** Sombras de cables/andamios, manchas de humedad/lluvia, suciedad/aceite y juntas constructivas.
* **Falsos negativos:** Fisuras con luz solar directa (sin sombra interna) y fisuras selladas con calcita blanca (eflorescencias).
* **Factores del dron:** Distancia de vuelo con $GSD > \text{ancho de fisura}$ (diluye la grieta) y desenfoque por vibración (*motion blur*).

### **3. Comparativa: OpenCV vs Hugging Face Spaces**
* **OpenCV:** Rápido y sin GPU, pero carece de contexto semántico; confunde manchas y sombras con fisuras (baja precisión en obra).
* **Hugging Face (Deep Learning):** Reconoce geometría y textura de falla; alta precisión e inmunidad a manchas, pero mayor costo computacional.

### **4. Criterio Estructural ante un 3% de Fisuración**
El 3% es una alerta de **tamizaje preliminar (screening 2D)**. Protocolo normativo (**ACI 224R / Eurocódigo 2**):
1. **Inspección visual:** Fisuras a $45^\circ$ en apoyos indican riesgo crítico de corte (apuntalamiento inmediato); verticales indican flexión; longitudinales indican corrosión de acero.
2. **Medición con fisurómetro:** Comparar abertura $w$ con límites normativos (ej. $w \le 0.30\text{ mm}$ a la intemperie según ACI 224R).
3. **Monitoreo:** Determinar si la fisura está activa (se mueve) o pasiva (estabilizada).
4. **Ensayos no destructivos:** Ultrasonido (UPV - ASTM C597) para profundidad y Pachometría/GPR para armaduras.
5. **Intervención:** Sellado hidrófugo (daño superficial), inyección epóxica a presión ASTM C881 (daño estructural) o refuerzo con fibra de carbono (CFRP).
